# OLIVES — Phase 0: Data Audit and Exploratory Analysis

**Project:** Clinically grounded, uncertainty-aware multimodal retinal biomarker detection
**Dataset:** OLIVES (Prabhushankar et al., NeurIPS 2022 Datasets & Benchmarks)

This notebook is the analysis front-end for the `olives_biomarkers` package. All logic lives in
`src/olives_biomarkers/` as importable classes; the notebook calls them and presents results.
Edit a `.py` file, re-run a cell, and the change is picked up (`autoreload` is on).

### What this notebook establishes

| # | Section | Question it answers |
|---|---------|--------------------|
| 1 | Environment & config | Where are we running, where is the data, what will be predicted? |
| 2 | Schema resolution | Do the declared columns exist verbatim in the parquet? |
| 3 | Manifest | Can we analyse 78k scans without touching 30 GB of image bytes? |
| 4 | Automated audit | What blocks or constrains modelling? |
| 5 | Cohort structure | How much *independent* evidence is there really? |
| 6 | Longitudinal structure | How do repeated visits constrain the split? |
| 7 | Duplicates | How much of the data is byte-identical repetition? |
| 8 | Label analysis | What is actually being predicted, and how imbalanced is it? |
| 9 | Label concentration | Which labels cannot support a patient-grouped test set? |
| 10 | Clinical variables | Do BCVA and CST carry biomarker signal at all? |
| 11 | Images | Are the scans clean, uniform, and correctly normalised? |
| 12 | Split design | What partitioning is defensible, and what does it cost? |
| 13 | Findings | What the experimental design must respect. |

> **Scope note.** The local copy is the Hugging Face parquet mirror. It has no fundus images,
> no 3D volumes, and no visit/week column — so the fundus and volume-level extensions in the
> project brief are blocked, and visit indices here are *inferred*. Section 2 makes this explicit.

---
## 1. Environment, configuration and imports

`RuntimeEnvironment` detects whether we are local or on Colab and resolves the data root
accordingly, so the same notebook runs in both places without edits.

In [1]:
%pip install -U ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.0/626.0 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 136.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 11.2 MB/s eta 0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 require

In [2]:
%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path

# Resolve the repo root whether the notebook runs from notebooks/ or the repo root.
REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(REPO_ROOT / "src"))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print("repo root:", REPO_ROOT)

repo root: /content


In [3]:
from olives_biomarkers.pipeline import OlivesPipeline
from olives_biomarkers.eda import OlivesEDA, EDAPlotter
from olives_biomarkers.data.dataset import ParquetImageReader

pipeline = OlivesPipeline.from_config(REPO_ROOT / "configs" / "data.yaml", repo_root=REPO_ROOT)
print(pipeline.describe())

ModuleNotFoundError: No module named 'olives_biomarkers'

### Colab

On Colab, mount Drive and point `data.root` at the uploaded copy. Section 11 shows how to export
a 2.3 GB image cache (just the modelling subset) rather than moving all 30 GB.

```python
pipeline.env.mount_drive()          # no-op when running locally
```

In [ ]:
# Where figures are written.
FIGURES = REPO_ROOT / "outputs" / "figures" / "eda"
FIGURES.mkdir(parents=True, exist_ok=True)

plotter = EDAPlotter()
SAVE_FIGURES = True

def show(figure, name: str):
    """Display a figure and optionally persist it to outputs/figures/eda."""
    if figure is None:
        print(f"(no figure produced for {name})")
        return
    if SAVE_FIGURES:
        plotter.save(figure, FIGURES / f"{name}.png")
    plt.show()

---
## 2. Schema resolution

The parquet column names are awkward (`"Atrophy / thinning of retinal layers"`,
`"Fluid (IRF)"`, `"Scan (n/49)"`). `configs/label_schema.yaml` maps each one to a stable
snake_case key **once**, and `LabelSchema` validates that every declared column really exists
before anything else runs.

The schema file also records the fields the project brief assumes but this data source does
*not* provide, so they are reported as unresolved rather than quietly invented.

In [ ]:
print(pipeline.schema.describe())

In [ ]:
# Fields the brief expects that are absent here - stated explicitly, not assumed away.
pd.DataFrame(
    [
        {"field": name, "status": spec.get("status"), "note": " ".join(str(spec.get("note", "")).split())}
        for name, spec in pipeline.schema.unresolved_fields.items()
    ]
)

---
## 3. Building the manifest

Images are stored inline in the parquet, so the two configs occupy ~30 GB. Almost every question
below is about scalar columns, so `ManifestBuilder` makes **one** pass that keeps the metadata and
a hash of each image, and discards the bytes. The result is a few megabytes.

The hash is what makes duplicate detection possible at all — without it we could not tell a
repeated visit from a genuinely new one.

First run: ~90 s. Afterwards it loads from cache in under a second.

In [ ]:
manifest = pipeline.get_manifest(split="train")   # rebuild=True to force a rescan
print(manifest)
print()
for key, value in manifest.summary().items():
    print(f"{key:>26}: {value}")

In [ ]:
manifest.frame.head(3)

In [ ]:
# Column inventory: what the manifest carries and how complete each column is.
inventory = pd.DataFrame(
    {
        "dtype": manifest.frame.dtypes.astype(str),
        "n_missing": manifest.frame.isna().sum(),
        "pct_present": (100 * manifest.frame.notna().mean()).round(2),
        "n_unique": manifest.frame.nunique(),
    }
)
inventory

**Read the `pct_present` column carefully.** `scan_number` is present on only ~22% of rows: it
belongs to the biomarker annotation block, not to every scan. It therefore cannot be used as a
general ordering key, and modelling rows must be filtered on `has_biomarkers` instead.

---
## 4. Automated audit

`DataAuditor` runs every Phase 0 check and grades findings as **blocker**, **warning** or **note**.
It writes `outputs/reports/data_audit.md` and `.json` so the same numbers can be cited later
without re-running anything.

In [ ]:
report = pipeline.run_audit(manifest, write=True)

blockers = [f for f in report.findings if f.severity == "blocker"]
warnings_ = [f for f in report.findings if f.severity == "warning"]
notes = [f for f in report.findings if f.severity == "info"]

print(f"{len(blockers)} blockers | {len(warnings_)} warnings | {len(notes)} notes\n")
for finding in report.findings:
    print(f"  {finding}\n")

In [ ]:
# The audit's own section tables.
report.sections["label_prevalence"]

---
## 5. Cohort structure — how much independent evidence exists

The headline count (78k scans) is misleading. Three views matter:

- **all rows** — everything in the parquet, duplicates included
- **unique images** — after removing byte-identical duplicates
- **biomarker-labelled** — deduplicated *and* carrying a complete 16-label vector; this is what a
  supervised model can actually train on

The gap between the first and last number is the single most important fact in this notebook.

In [ ]:
eda = OlivesEDA(manifest, dedup_policy="keep_first")
eda.cohort_overview()

The modelling set should come out near **9,400 labelled scans from 87 patients** — very close to
the 9,408 the OLIVES paper reports, which is a good independent check that the deduplication is
right.

Note the visit count in the labelled row: ~192 visits over 96 eyes, i.e. **exactly two per eye**.
That matches the paper's protocol — biomarkers were graded only on each eye's first and last visit.

In [ ]:
patients = eda.per_patient_summary()
display(patients.head(10))
patients[["n_scans", "n_eyes", "n_visits", "n_labelled_scans", "bcva_mean", "cst_mean"]].describe().round(2)

In [ ]:
show(plotter.per_patient_distribution(patients), "05_per_patient_distribution")

In [ ]:
eda.disease_distribution()

DR and DME are close to balanced at patient level, which is convenient: the split can stratify on
disease without distorting the partitions. (In the source trials, DR comes from PRIME and DME from
TREX-DME, so "disease" is partly a proxy for "trial".)

---
## 6. Longitudinal structure

This mirror has **no visit or week column**. `VisitInferencer` reconstructs visits from the fact
that BCVA and CST are measured once per visit and stay constant across that visit's 49 B-scans:
a new visit opens when `(bcva, cst)` changes, or when 49 scans have accumulated.

These indices are **inferred**, not authoritative. They are good enough to characterise the
repeated-measures structure — which is what governs the split — but no visit-level *result*
should be reported without manual verification.

In [ ]:
visits = eda.visits_per_eye()
print(f"{len(visits)} eyes, {visits['n_visits'].sum()} inferred visits")
print(visits["n_visits"].describe().round(1))

figure, axis = plt.subplots(figsize=(9, 4))
axis.hist(visits["n_visits"], bins=range(1, int(visits["n_visits"].max()) + 2), color="#4C72B0")
axis.axvline(visits["n_visits"].median(), color="#C44E52", ls="--",
             label=f"median {visits['n_visits'].median():.0f}")
axis.set_xlabel("inferred visits per eye"); axis.set_ylabel("eyes")
axis.set_title("Visits per eye"); axis.legend()
show(figure, "06_visits_per_eye")

In [ ]:
trajectories = eda.visit_trajectories()
trajectories.head(10)

In [ ]:
progression = eda.treatment_progression()
display(progression.head(15))
show(plotter.treatment_progression(progression), "06_treatment_progression")

This reproduces the shape of Figure 5 in the OLIVES paper: BCVA improves substantially against
baseline, while the *visit-to-visit* improvement rate decays. The attrition panel explains why —
the cohort thins with each visit, and the patients who keep returning are the harder ones, which
drags the later averages down. It is survivorship, not deterioration.

In [ ]:
variability = eda.within_eye_variability()
display(variability.describe().round(2))

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(variability["bcva_range"].dropna(), bins=30, color="#4C72B0")
axes[0].set_xlabel("BCVA range within one eye"); axes[0].set_ylabel("eyes")
axes[0].set_title("Within-eye BCVA spread across visits")
axes[1].hist(variability["cst_range"].dropna(), bins=30, color="#DD8452")
axes[1].set_xlabel("CST range within one eye")
axes[1].set_title("Within-eye CST spread across visits")
figure.tight_layout()
show(figure, "06_within_eye_variability")

In [ ]:
# Biomarkers are graded only at the first and last visit - so this is the treatment contrast.
eda.first_last_visit_comparison()

The first-versus-last contrast is the treatment effect the paper describes as a *domain shift*:
models trained on first-visit scans transfer poorly to last-visit scans. Two consequences for us:
a split must not place one eye's first visit in train and its last visit in test, and any
"before/after treatment" comparison is a separate experiment, not a free result.

---
## 7. Duplicate images

The dataset card warns that some visits repeat an earlier visit's scans (it names patient 61,
W8 vs W12). Hashing every image quantifies it.

Two distinct phenomena, which need different treatment:

- **adjacent duplicates** — the same row emitted twice in succession; a storage artefact
- **repeated-visit duplicates** — a later visit whose images are byte-identical to an earlier one;
  a genuine clinical time point that happens to carry the same pixels

In [ ]:
duplicates = eda.duplicate_analysis()
pd.Series(duplicates).to_frame("value")

In [ ]:
show(plotter.duplicate_summary(duplicates), "07_duplicate_composition")

In [ ]:
# Which eyes have visits that repeat an earlier visit's images.
eda.duplicate_visits()

About **21% of rows are byte-identical to another row**. Crucially, no duplicate group spans two
different patients, so patient-grouped splitting already contains them — but they still have to be
removed before counting, or every prevalence figure is inflated by repetition.

`dedup_policy: keep_first` in `configs/data.yaml` controls this; the modelling frame applies it.

---
## 8. Label analysis — what is actually being predicted

In [ ]:
prevalence = eda.label_prevalence()
prevalence[["label", "display", "n_labelled", "n_positive", "pct", "n_patients_positive", "rarity"]]

In [ ]:
show(plotter.label_prevalence(prevalence), "08_label_prevalence")

The imbalance spans **three orders of magnitude**: IRHRF appears on roughly two thirds of scans,
while PED (serous), VMT and RPE disruption appear on well under 1%. A single global 0.5 threshold
cannot serve both ends of that range, which is why the pipeline fits one threshold per label on
validation data.

In [ ]:
cardinality = eda.label_cardinality()
pd.Series(cardinality).to_frame("value")

In [ ]:
show(plotter.label_cardinality(eda.label_cardinality_distribution()), "08_label_cardinality")

Scans carry roughly **3 biomarkers on average**, and only 292 of the 65,536 possible label vectors
ever occur. The task is genuinely multilabel with strong structure between labels — not 16
independent binary problems — which is what makes exact-match a harsh but informative metric.

In [ ]:
show(plotter.cooccurrence_heatmap(eda.label_cooccurrence_matrix("jaccard")), "08_cooccurrence_jaccard")

In [ ]:
show(plotter.correlation_heatmap(eda.label_correlation()), "08_label_correlation")

Look for the **PAVF / FAVF** pair: partially- and fully-attached vitreous face are near-mutually
exclusive by definition, so they should show strong negative correlation. If a trained model
predicts both as present on the same scan, that is a concrete, checkable failure.

In [ ]:
eda.top_label_combinations(top_n=15)

In [ ]:
by_disease = eda.label_prevalence_by_disease()
display(by_disease)
show(plotter.prevalence_by_disease(by_disease), "08_prevalence_by_disease")

Biomarkers overlap heavily across DR and DME — exactly the paper's point that *"biomarkers in
isolation are insufficient to diagnose disease states"*. Useful as a sanity check, and a reason not
to feed the disease label to a biomarker model: it would leak trial identity.

---
## 9. Label concentration and split feasibility

A label with 10 positives is not merely hard to learn — if those positives sit inside one or two
patients, a patient-grouped test set will frequently contain **zero** of them, and the per-label
metric is then undefined rather than bad.

In [ ]:
concentration = eda.label_patient_concentration()
concentration

In [ ]:
show(plotter.patient_label_heatmap(eda.label_prevalence_by_patient()), "09_patient_label_heatmap")

The heatmap shows how patient-specific the rare labels are: solid columns mean a biomarker is a
property of a *patient*, not of an individual scan. For those labels the model can succeed simply
by recognising the patient, which patient-grouped splitting correctly makes impossible.

In [ ]:
feasibility = eda.split_feasibility(test_fraction=0.15)
display(feasibility)
show(plotter.split_feasibility(feasibility), "09_split_feasibility")

**This table drives the target-set decision.** Labels marked `usable_in_test = False` cannot be
evaluated at a 15% patient holdout. The 6-label MVP (`target_set: six` — IRHRF, PAVF, FAVF, IRF,
DRT/ME, vitreous debris) is precisely the subset that survives, which is why the VIP Cup chose it.

Reporting the 16-label model is still worthwhile, but per-label results for the rare labels must be
reported as undefined rather than as a number.

---
## 10. Clinical variables — do BCVA and CST carry biomarker signal?

This is the crux of the research question. If BCVA and CST separate biomarkers, clinical fusion has
something to work with; if they do not, the honest result is a negative one.

In [ ]:
eda.clinical_summary()

In [ ]:
show(plotter.clinical_distributions(eda.unique, by_disease=True), "10_clinical_distributions")

In [ ]:
eda.clinical_by_disease()

In [ ]:
# Missingness: concentrated, not scattered.
eda.missingness_profile()

Missing BCVA/CST is confined to **one patient (79)**, as the dataset card warns. Because it is
concentrated rather than random, the missingness indicator is itself informative — dropping those
rows would silently remove one patient from the cohort, so the pipeline imputes on the training
fold and keeps an explicit indicator.

In [ ]:
association = eda.clinical_label_association()
association[["label", "n_present", "n_absent", "cst_present", "cst_absent",
             "cst_delta", "cst_cohens_d", "bcva_delta", "bcva_cohens_d"]]

In [ ]:
show(plotter.clinical_label_association(association, "cst"), "10_cst_association")
show(plotter.clinical_label_association(association, "bcva"), "10_bcva_association")

In [ ]:
eda.clinical_correlation()

In [ ]:
show(plotter.clinical_scatter(eda.labelled, label="irf"), "10_clinical_scatter_irf")
show(plotter.clinical_scatter(eda.labelled, label="drt_me"), "10_clinical_scatter_drtme")

**Interpretation.** CST is macular thickness, so it should separate the fluid and thickening
biomarkers (IRF, DRT/ME) and be largely uninformative about vitreous-face biomarkers (PAVF, FAVF)
— those concern structure above the retina that CST does not measure. Check that pattern above:
if it holds, it is mechanistic support for gating rather than an accident of correlation.

Note the ceiling on what fusion can achieve: CST is one number per *volume*, so it is identical
across all 49 B-scans of a visit while the biomarkers vary slice to slice. Clinical features can
therefore shift a scan's prior, but can never explain within-volume variation. That is a real
limit on the effect size to expect from Models C and D.

---
## 11. Image-level analysis

In [ ]:
reader = ParquetImageReader(pipeline.data_root, pipeline.config.data.config_name, "train")
image_stats = eda.image_statistics(reader, n_samples=300, seed=42)
image_stats.head()

In [ ]:
summary = eda.image_statistics_summary(image_stats)
pd.Series(summary).to_frame("value")

### Two findings that change the preprocessing

1. **Resolution is uniform** (a single `width x height`, mode `L`), so no aspect-ratio handling is
   needed — a plain resize is safe.
2. **The paper's normalisation constants do not match this data.** The paper cites
   `mean = 0.482, std = 0.037`; the measured pixel mean here is far lower, because roughly half of
   each B-scan is near-black vitreous above the retina. A `std` of 0.037 is also implausible for
   raw pixels — it looks like the standard deviation of per-image *means*, not of pixels.

   Using those constants verbatim would badly mis-scale the input. `ImageTransformFactory`
   therefore defaults to ImageNet normalisation with a 3-channel stem (matching the pretrained
   encoder), and the measured statistics above are what a 1-channel variant should use.

In [ ]:
show(plotter.image_statistics(image_stats), "11_image_statistics")

In [ ]:
show(plotter.image_grid(eda.sample_images(reader, n=8, seed=7), n_cols=4,
                        title="Random OCT B-scans"), "11_sample_random")

In [ ]:
# Contrast scans with and without a biomarker - is it visible to the eye?
for label in ["irf", "drt_me"]:
    show(plotter.image_grid(eda.sample_images(reader, n=4, label=label, present=True, seed=3),
                            n_cols=4, title=f"{label} PRESENT"), f"11_sample_{label}_present")
    show(plotter.image_grid(eda.sample_images(reader, n=4, label=label, present=False, seed=3),
                            n_cols=4, title=f"{label} ABSENT"), f"11_sample_{label}_absent")

### Exporting an image cache (needed for Colab)

Training does not need all 30 GB — only the 9,396 labelled, deduplicated scans. Exporting them as
PNGs gives a folder small enough to upload to Drive and fast enough for random access.

Run once, locally, then upload `data/processed/images_labelled/`.

In [ ]:
from olives_biomarkers.data.dataset import ImageCacheExporter

EXPORT_IMAGES = False   # set True to run the one-off export

if EXPORT_IMAGES:
    exporter = ImageCacheExporter(
        pipeline.data_root,
        pipeline.config.data.config_name,
        output_dir=REPO_ROOT / "data" / "processed" / "images_labelled",
        split="train",
    )
    cached = exporter.export(eda.labelled)
    print(f"cache size: {exporter.cache_size_gb():.2f} GB")
    cached[["row_uid", "patient_id", "cache_path"]].head()
else:
    print("EXPORT_IMAGES is False - set it to True to build the Colab-friendly image cache.")

---
## 12. Split design

Everything above converges on one constraint: **partition patients, never scans**.

In [ ]:
eda.leakage_risk_summary()

Read the last column as the honest sample size. Splitting at the scan level would suggest tens of
thousands of independent observations; the truth is **87**. A random image-level split would place
49 near-identical slices of one volume on both sides and report an accuracy that means nothing.

In [ ]:
assignment = pipeline.make_holdout_split(manifest, write=True)
pd.DataFrame(
    {
        "partition": list(assignment.partitions),
        "n_patients": [len(v) for v in assignment.partitions.values()],
    }
)

In [ ]:
from olives_biomarkers.data.splits import SplitManifestWriter

modelling = manifest.modelling_frame(policy="keep_first", labelled_only=True)
writer = SplitManifestWriter(pipeline.split_dir)
writer.prevalence_by_partition(modelling, assignment, manifest.label_columns)

Compare prevalence across partitions. Common labels should track closely; rare ones will swing
wildly or read 0.0 in some partitions — a limitation to report explicitly, not to fix by
reshuffling until it looks good.

The `calibration` partition is carved out of *train* and is patient-disjoint from everything else,
so temperature scaling never sees data that fitted the model weights.

In [ ]:
folds = pipeline.make_folds(manifest, write=True)
pd.DataFrame(
    [
        {
            "fold": f.name,
            "train": len(f.train),
            "val": len(f.val),
            "calibration": len(f.calibration),
            "test": len(f.test),
        }
        for f in folds
    ]
)

In [ ]:
# Verify disjointness rather than trusting it.
from olives_biomarkers.data.splits import SplitValidator

validator = SplitValidator()
for fold in [assignment, *folds]:
    validator.validate(modelling, fold)
print(f"all {1 + len(folds)} splits verified: patients disjoint, duplicate groups contained")

---
## 13. Findings

In [ ]:
for i, finding in enumerate(eda.key_findings(), 1):
    print(f"{i}. {finding}\n")

### What this means for the experimental design

**Settled by the data**

1. **Patient-grouped splitting is mandatory.** 87 patients, ~108 labelled scans each. The effective
   sample size is the patient count; every confidence interval must be a patient-level bootstrap.
2. **Deduplicate before anything else.** ~21% of rows are byte-identical repeats. They inflate
   every count and would corrupt any scan-level split.
3. **`has_biomarkers` is the filter for modelling rows** — not `scan_number`, which is present on
   only 22% of rows.
4. **Per-label thresholds, not 0.5.** Prevalence ranges from 67% to 0.06%.
5. **Report the 6-label MVP as the primary result**, with the 16-label model secondary and the rare
   labels reported as undefined where the test fold holds no positives.
6. **Do not use the paper's normalisation constants.** They do not match the pixel statistics here.

**Open questions the modelling has to answer**

7. Does CST's association with fluid biomarkers survive as a *conditional* effect once the image is
   in the model? Section 10 shows only a marginal association.
8. Can clinical features help at all at slice level, given one CST value covers all 49 slices of a
   volume while the labels vary between them? This bounds the plausible effect size for Models C
   and D — and makes a well-controlled negative result a legitimate outcome.

**Blocked**

9. Fundus and 3D-volume extensions: not possible with this mirror. They need the Zenodo release.
10. Visit-level results: visit indices are inferred and not authoritative.

### Next step

```bash
python scripts/audit_data.py --config configs/data.yaml     # reproduce sections 3-4
python scripts/build_manifest.py --config configs/data.yaml
python scripts/make_splits.py --config configs/data.yaml
```

Then Phase 2 — the clinical-only, OCT-only and concatenation baselines — using the fold manifests
written in section 12.